<a href="https://colab.research.google.com/github/andehnoun/AutoML-Homework/blob/main/H2O_AutoML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installer les outils AutoML

In [4]:
!pip -q install h2o


In [5]:
import h2o
h2o.init()
print("H2O OK ✅")


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.17" 2025-10-21; OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04); OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpkoke_vtg
  JVM stdout: /tmp/tmpkoke_vtg/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpkoke_vtg/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,1 month and 18 days
H2O_cluster_name:,H2O_from_python_unknownUser_g1taok
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.168 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


H2O OK ✅


Charger le dataset

In [6]:
import pandas as pd

df = pd.read_csv("car_price_prediction_with_missing.csv")
df.head()


,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1.0,Tesla,2016.0,2.3,Petrol,Manual,114832.0,New,26613.92,Model X
1,2.0,BMW,2018.0,4.4,Electric,Manual,143190.0,Used,14679.61,5 Series
2,3.0,Audi,2013.0,4.5,Electric,Manual,181601.0,New,44402.61,A4
3,4.0,Tesla,2011.0,4.1,Diesel,Automatic,68682.0,New,86374.33,Model Y
4,5.0,Ford,2009.0,2.6,Diesel,Manual,223009.0,Like New,73577.10,Mustang


Compréhension des données

In [7]:
df.info()
df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car ID        2250 non-null   float64
 1   Brand         2250 non-null   object 
 2   Year          2250 non-null   float64
 3   Engine Size   2250 non-null   float64
 4   Fuel Type     2250 non-null   object 
 5   Transmission  2250 non-null   object 
 6   Mileage       2250 non-null   float64
 7   Condition     2250 non-null   object 
 8   Price         2250 non-null   float64
 9   Model         2250 non-null   object 
dtypes: float64(5), object(5)
memory usage: 195.4+ KB


,0
Car ID,250
Brand,250
Year,250
Engine Size,250
Fuel Type,250
Transmission,250
Mileage,250
Condition,250
Price,250
Model,250


Le dataset concerne la prédiction du prix des voitures. Il contient des variables numériques (année, taille du moteur, kilométrage, prix) et des variables catégorielles (marque, type de carburant, transmission, modèle, état). L’objectif est d’utiliser une approche AutoML pour entraîner automatiquement plusieurs modèles et sélectionner le meilleur afin de prédire la variable cible Price.

Nettoage des données

In [8]:
df = df.drop(columns=["Car ID"])
df.head()


,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,Tesla,2016.0,2.3,Petrol,Manual,114832.0,New,26613.92,Model X
1,BMW,2018.0,4.4,Electric,Manual,143190.0,Used,14679.61,5 Series
2,Audi,2013.0,4.5,Electric,Manual,181601.0,New,44402.61,A4
3,Tesla,2011.0,4.1,Diesel,Automatic,68682.0,New,86374.33,Model Y
4,Ford,2009.0,2.6,Diesel,Manual,223009.0,Like New,73577.10,Mustang


La colonne Car ID a été supprimée car il s’agit uniquement d’un identifiant et qu’elle n’apporte aucune information utile pour la prédiction du prix. La variable cible du problème est Price, ce qui correspond à un problème de régression.

In [11]:
h2o_df = h2o.H2OFrame(df)



Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


Définir la cible et les variables

In [12]:
y = "Price"
x = h2o_df.columns
x.remove(y)

print("Nombre de variables explicatives :", len(x))
print("Cible :", y)


Nombre de variables explicatives : 8
Cible : Price


Séparer les données train/ test

In [13]:
train, test = h2o_df.split_frame(ratios=[0.8], seed=42)

print("Taille du jeu d'entraînement :", train.nrows)
print("Taille du jeu de test :", test.nrows)


Taille du jeu d'entraînement : 2002
Taille du jeu de test : 498


Lancer H2O AutoML

In [14]:
from h2o.automl import H2OAutoML

aml = H2OAutoML(
    max_models=10,
    max_runtime_secs=300,  # 5 minutes
    seed=42
)

aml.train(x=x, y=y, training_frame=train)


AutoML progress: |
22:14:34.640: XGBoost_1_AutoML_1_20260111_221433 [XGBoost def_2] failed: water.exceptions.H2OModelBuilderIllegalArgumentException: Illegal argument(s) for XGBoost model: XGBoost_1_AutoML_1_20260111_221433_cv_1.  Details: ERRR on field: _response_column: Response contains missing values (NAs) - not supported by XGBoost.


████
22:14:40.128: XGBoost_2_AutoML_1_20260111_221433 [XGBoost def_1] failed: water.exceptions.H2OModelBuilderIllegalArgumentException: Illegal argument(s) for XGBoost model: XGBoost_2_AutoML_1_20260111_221433_cv_1.  Details: ERRR on field: _response_column: Response contains missing values (NAs) - not supported by XGBoost.


███████
22:14:56.872: XGBoost_3_AutoML_1_20260111_221433 [XGBoost def_3] failed: water.exceptions.H2OModelBuilderIllegalArgumentException: Illegal argument(s) for XGBoost model: XGBoost_3_AutoML_1_20260111_221433_cv_1.  Details: ERRR on field: _response_column: Response contains missing values (NAs) - not supported by XGBoost.



key,value
Stacking strategy,cross_validation
Number of base models (used / total),0/5
# GBM base models (used / total),0/1
# GLM base models (used / total),0/1
# DeepLearning base models (used / total),0/1
# DRF base models (used / total),0/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [15]:
df_clean = df.dropna(subset=["Price"])
df_clean.shape


(2250, 9)

Lors de l’exécution d’AutoML, certains modèles (notamment XGBoost) ont échoué car la variable cible Price contenait des valeurs manquantes. Comme la prédiction du prix nécessite une cible définie, les observations dont Price est manquant ont été supprimées avant de relancer H2O AutoML. Cette étape améliore la qualité de l’apprentissage et permet à tous les algorithmes de fonctionner correctement.

Relancer H2O AutoML proprement : Reconvertir en H2OFrame + train/test

In [16]:
h2o_df2 = h2o.H2OFrame(df_clean)

y = "Price"
x = h2o_df2.columns
x.remove(y)

train2, test2 = h2o_df2.split_frame(ratios=[0.8], seed=42)

print("Train:", train2.nrows, "Test:", test2.nrows)


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Train: 1793 Test: 457


Relancer H2O AutoML (version corrigée)

In [18]:
from h2o.automl import H2OAutoML

aml2 = H2OAutoML(
    max_models=10,
    max_runtime_secs=300,  # 5 minutes
    seed=42
)

aml2.train(x=x, y=y, training_frame=train2)


AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGeneralizedLinearEstimator : Generalized Linear Modeling
Model Key: GLM_1_AutoML_3_20260111_223323


GLM Model: summary
    family    link      regularization              lambda_search                                                                     number_of_predictors_total    number_of_active_predictors    number_of_iterations    training_frame
--  --------  --------  --------------------------  --------------------------------------------------------------------------------  ----------------------------  -----------------------------  ----------------------  -----------------------------------------------
    gaussian  identity  Ridge ( lambda = 25612.0 )  nlambda = 30, lambda.max = 106940.0, lambda.min = 25612.0, lambda.1se = 106940.0  47                            47                             4                       AutoML_3_20260111_223323_training_py_8_sid_80de

ModelMetricsRegressionGLM: glm
** Reported on train data. **

MSE: 729731462.8224822
RMSE: 27013.542211684904
MAE: 23422.074892347602
RMSLE: 0.7171180403826323
Mean Residual Deviance: 729731462.8224822
R^2: 3.6286197957480226e-07
Null degrees of freedom: 1792
Residual degrees of freedom: 1745
Null deviance: 1308408987612.589
Residual deviance: 1308408512840.7107
AIC: 41778.19316742376

ModelMetricsRegressionGLM: glm
** Reported on cross-validation data. **

MSE: 729955464.1921785
RMSE: 27017.687987542133
MAE: 23425.550442356394
RMSLE: 0.7171826148845922
Mean Residual Deviance: 729955464.1921785
R^2: -0.0003066011375547628
Null degrees of freedom: 1792
Residual degrees of freedom: 1745
Null deviance: 1308810241942.197
Residual deviance: 1308810147296.576
AIC: 41778.743469617446

Cross-Validation Metrics Summary: 
                        mean          sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  ------------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     8433.98       15.9952      8432.53       8459.94       8432.6        8416.18       8428.65
loglikelihood           0             0            0             0             0             0             0
mae                     23425.6       472.555      23025.4       24138.1       23091.2       23198.4       23674.6
mean_residual_deviance  7.29961e+08   2.49948e+07  7.08097e+08   7.64282e+08   7.08226e+08   7.21802e+08   7.47396e+08
mse                     7.29961e+08   2.49948e+07  7.08097e+08   7.64282e+08   7.08226e+08   7.21802e+08   7.47396e+08
null_deviance           2.61762e+11   8.9105e+09   2.54207e+11   2.74377e+11   2.54253e+11   2.58405e+11   2.67568e+11
r2                      -0.000874005  0.00106663   -0.00217292   -0.000165794  -0.00190062   -5.39039e-05  -7.67872e-05
residual_deviance       2.61762e+11   8.91049e+09  2.54207e+11   2.74377e+11   2.54253e+11   2.58405e+11   2.67568e+11
rmse                    27014.6       461.098      26610.1       27645.7       26612.5       26866.4       27338.5
rmsle                   0.716879      0.0233414    0.684822      0.749061      0.715161      0.710203      0.72515

Scoring History: 
    timestamp            duration    iteration    lambda    predictors    deviance_train    deviance_xval    deviance_se    alpha    iterations    training_rmse       training_deviance    training_mae        training_r2
--  -------------------  ----------  -----------  --------  ------------  ----------------  ---------------  -------------  -------  ------------  ------------------  -------------------  ------------------  ----------------------
    2026-01-11 22:33:26  0.000 sec   1            110000    48            7.29732e+08       7.29961e+08      1.1178e+07     0
    2026-01-11 22:33:26  0.000 sec   2            66000     48            7.29732e+08       7.29961e+08      1.1178e+07     0
    2026-01-11 22:33:26  0.001 sec   3            41000     48            7.29732e+08       7.29961e+08      1.1178e+07     0
    2

Après suppression des lignes dont la variable cible Price était manquante, H2O AutoML a été relancé sur un ensemble d’apprentissage (80 %) et un ensemble de test (20 %). AutoML a entraîné automatiquement plusieurs modèles de régression et a sélectionné le meilleur modèle (leader). Dans cette exécution, le meilleur modèle identifié est un modèle de type GLM (régression linéaire régularisée), et l’importance des variables suggère que l’année, le kilométrage et le type de carburant contribuent fortement à la prédiction du prix.

In [19]:
lb = aml2.leaderboard
lb.head(10)


model_id,rmse,mse,mae,rmsle,mean_residual_deviance
GLM_1_AutoML_3_20260111_223323,27017.7,7.29955e+08,23425.6,0.717183,7.29955e+08
StackedEnsemble_AllModels_1_AutoML_3_20260111_223323,27024.4,7.30318e+08,23431.7,0.717286,7.30318e+08
StackedEnsemble_BestOfFamily_1_AutoML_3_20260111_223323,27028.9,7.3056e+08,23424.2,0.717404,7.3056e+08
GBM_1_AutoML_3_20260111_223323,27251.7,7.42653e+08,23509.2,0.720325,7.42653e+08
GBM_2_AutoML_3_20260111_223323,27593.4,7.61395e+08,23648.3,0.723172,7.61395e+08
XRT_1_AutoML_3_20260111_223323,27659.5,7.6505e+08,23728.2,0.719136,7.6505e+08
GBM_4_AutoML_3_20260111_223323,27793.8,7.72495e+08,23722.8,0.72651,7.72495e+08
GBM_3_AutoML_3_20260111_223323,27854.2,7.75858e+08,23806.6,0.727611,7.75858e+08
DRF_1_AutoML_3_20260111_223323,27962.4,7.81894e+08,23771.8,0.725331,7.81894e+08
XGBoost_3_AutoML_3_20260111_223323,28577.4,8.1667e+08,24194,0.736915,8.1667e+08


Interprétation :
Le modèle GLM est classé premier du leaderboard, indiquant qu’un modèle simple régularisé est le plus adapté à ce dataset.

Dernière étape : Évaluer le meilleur modèle sur le jeu de test

In [20]:
best_model = aml2.leader

performance_test = best_model.model_performance(test2)
performance_test


ModelMetricsRegressionGLM: glm
** Reported on test data. **

MSE: 786820959.6399883
RMSE: 28050.329046911167
MAE: 24569.607150368065
RMSLE: 0.768884616624469
Mean Residual Deviance: 786820959.6399883
R^2: -0.0016118558483833034
Null degrees of freedom: 456
Residual degrees of freedom: 409
Null deviance: 359577153253.2433
Residual deviance: 359577178555.4747
AIC: 10755.874475780345

Le modèle sélectionné par H2O AutoML (GLM) a été évalué sur le jeu de test. Il obtient une RMSE d’environ 28 050 et une MAE d’environ 24 570. Le coefficient R² étant proche de zéro, les performances restent limitées, ce qui montre que la prédiction du prix des voitures est difficile avec les variables disponibles.